# PFE Navigator – Assistant intelligent basé sur RAG

## 1. Présentation du projet

PFE Navigator est un assistant intelligent qui répond aux questions
des étudiants concernant le Projet de Fin d'Études (PFE).

Il utilise une collection de documents PDF pour rechercher
les informations pertinentes et générer des réponses
en indiquant les sources utilisées.

## 2. Objectif

Construire un système RAG capable de :
- Lire des documents PDF.
- Extraire et nettoyer leur contenu.
- Découper les textes en morceaux (chunks).
- Retrouver les passages pertinents.
- Générer des réponses avec leurs sources.

In [1]:
import pypdf
import re
from pathlib import Path

print("Les bibliothèques sont prêtes !")

Les bibliothèques sont prêtes !


## 3. Chargement et extraction des documents PDF

Dans cette étape, nous allons parcourir le dossier documents,
ouvrir chaque fichier PDF et extraire son texte page par page.

Nous allons également supprimer les pieds de page commençant
par « ISIT – » afin d'éviter de répéter ces informations
dans les futurs chunks.

In [4]:
from pathlib import Path
from pypdf import PdfReader
import re

# Chemin vers le dossier contenant les PDF
documents_path = Path("../documents")

# Liste qui va contenir toutes les pages extraites
documents = []

# Parcourir tous les fichiers PDF
for pdf_file in documents_path.glob("*.pdf"):

    reader = PdfReader(pdf_file)

    print("Lecture du document :", pdf_file.name)

    # Parcourir chaque page
    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if text:

            lines = text.splitlines()

            # Supprimer les pieds de page
            lines = [
                line for line in lines
                if not line.strip().startswith("ISIT –")
            ]

            clean_text = "\n".join(lines)
            clean_text = re.sub(r"\n{2,}", "\n", clean_text)

            documents.append({
                "text": clean_text.strip(),
                "source": pdf_file.name,
                "page": page_number
            })

print("\nExtraction terminée !")
print("Nombre de pages extraites :", len(documents))
print("Nombre de documents PDF :", len(set(d["source"] for d in documents)))

Lecture du document : 01_guide_pfe.pdf
Lecture du document : 02_reglement_stages.pdf
Lecture du document : 03_guide_redaction.pdf
Lecture du document : 04_guide_soutenance.pdf
Lecture du document : 05_calendrier_universitaire.pdf

Extraction terminée !
Nombre de pages extraites : 13
Nombre de documents PDF : 5


In [5]:
#Afficher un extrait pour vérifier
for doc in documents[:3]:

    print("Document :", doc["source"])
    print("Page :", doc["page"])
    print("\nContenu :")
    print(doc["text"][:500])
    print("\n" + "=" * 60)

Document : 01_guide_pfe.pdf
Page : 1

Contenu :
Page 1
Guide du Projet de Fin d'Études (PFE) – ISIT –
2026-2027
Institut Supérieur d'Informatique et de Technologies (ISIT) · Version 3.0 · 2026-09-01
1. Définition et objectifs du PFE
Le Projet de Fin d'Études (PFE) est l'épreuve finale du cycle ingénieur de l'ISIT, réalisée en
troisième année. Il vaut 30 crédits ECTS et se déroule en entreprise ou en laboratoire de
recherche. Son objectif est de mettre l'étudiant en situation professionnelle : analyser un besoin,
concevoir, développer, tester 

Document : 01_guide_pfe.pdf
Page : 2

Contenu :
Page 2
L'étudiant doit envoyer à son encadrant académique un point d'avancement écrit toutes les deux
semaines.
7. Documents à fournir pendant le PFE
•
La convention de PFE, signée par les trois parties (étudiant, entreprise, ISIT) avant le
démarrage.
•
Un rapport de mi-parcours de 5 à 10 pages, à rendre avant le 15 avril 2027.
•
Le pré-rapport complet, à rendre avant le 24 mai 2027 pour relecture p

In [6]:
# Vérifier le début et la fin du texte de quelques pages

for doc in documents[:3]:

    print("Document :", doc["source"])
    print("Page :", doc["page"])

    print("\nDébut du texte :")
    print(doc["text"][:250])

    print("\nFin du texte :")
    print(doc["text"][-250:])

    print("\n" + "=" * 60)

Document : 01_guide_pfe.pdf
Page : 1

Début du texte :
Page 1
Guide du Projet de Fin d'Études (PFE) – ISIT –
2026-2027
Institut Supérieur d'Informatique et de Technologies (ISIT) · Version 3.0 · 2026-09-01
1. Définition et objectifs du PFE
Le Projet de Fin d'Études (PFE) est l'épreuve finale du cycle ing

Fin du texte :
e scientifique, relit le rapport et organise au moins 4 réunions de suivi.
•
L'encadrant professionnel est un ingénieur ou un cadre de la structure d'accueil, ayant au
moins 3 ans d'expérience. Il définit les tâches au quotidien et évalue l'étudiant.

Document : 01_guide_pfe.pdf
Page : 2

Début du texte :
Page 2
L'étudiant doit envoyer à son encadrant académique un point d'avancement écrit toutes les deux
semaines.
7. Documents à fournir pendant le PFE
•
La convention de PFE, signée par les trois parties (étudiant, entreprise, ISIT) avant le
démarrage

Fin du texte :
me : exceptionnel, autorisé seulement si le sujet est assez large ; chaque étudiant
rédige alors son pro

In [7]:
# Statistiques de l'extraction

print("Nombre total de pages extraites :", len(documents))

print("\nNombre de pages par document :")

from collections import Counter

pages_par_document = Counter(
    doc["source"] for doc in documents
)

for nom, nombre in pages_par_document.items():
    print(nom, ":", nombre, "pages")

Nombre total de pages extraites : 13

Nombre de pages par document :
01_guide_pfe.pdf : 2 pages
02_reglement_stages.pdf : 3 pages
03_guide_redaction.pdf : 3 pages
04_guide_soutenance.pdf : 3 pages
05_calendrier_universitaire.pdf : 2 pages


Étape 3.2 — Nettoyage des documents

In [8]:


import re

for doc in documents:

    texte = doc["text"]

    lignes = texte.splitlines()
    lignes_nettoyees = []

    for ligne in lignes:

        # Supprimer toute ligne contenant le pied de page ISIT
        if re.search(r'ISIT\s*[–—-]', ligne):
            continue

        lignes_nettoyees.append(ligne)

    texte_nettoye = "\n".join(lignes_nettoyees)

    # Réduire les espaces multiples
    texte_nettoye = re.sub(r'[ \t]+', ' ', texte_nettoye)

    # Réduire les lignes vides multiples
    texte_nettoye = re.sub(r'\n{3,}', '\n\n', texte_nettoye)

    doc["text"] = texte_nettoye.strip()

print("Nettoyage corrigé terminé !")


Nettoyage corrigé terminé !


In [9]:
# Vérifier le texte après nettoyage

for doc in documents[:3]:

    print("Document :", doc["source"])
    print("Page :", doc["page"])
    print("\nContenu nettoyé :")
    print(doc["text"][-300:])
    print("\n" + "=" * 60)

Document : 01_guide_pfe.pdf
Page : 1

Contenu nettoyé :
lus tard le 16 novembre
2026. Il valide la démarche scientifique, relit le rapport et organise au moins 4 réunions de suivi.
•
L'encadrant professionnel est un ingénieur ou un cadre de la structure d'accueil, ayant au
moins 3 ans d'expérience. Il définit les tâches au quotidien et évalue l'étudiant.

Document : 01_guide_pfe.pdf
Page : 2

Contenu nettoyé :
, avec une assurance
internationale.
•
PFE en binôme : exceptionnel, autorisé seulement si le sujet est assez large ; chaque étudiant
rédige alors son propre rapport sur sa partie.
11. Contacts
Service des Stages et PFE : stages@isit.example.tn, bureau B-104, ouvert du lundi au vendredi de
9h à 16h.

Document : 02_reglement_stages.pdf
Page : 1

Contenu nettoyé :
0 heures par semaine, selon les horaires de l'entreprise. Le
stagiaire respecte le règlement intérieur de la structure d'accueil. Toute absence doit être justifiée
auprès de l'encadrant professionnel. Au-delà de 3 jours d

In [10]:
# Vérifier si les pieds de page ISIT sont encore présents

for doc in documents:
    if "ISIT –" in doc["text"]:
        print("Pied de page détecté :", doc["source"], "Page", doc["page"])

print("Vérification terminée.")

Vérification terminée.


Étape 3.3 — Découpage en chunks

In [18]:

# Paramètres du découpage
chunk_size = 500
chunk_overlap = 100

chunks = []

for doc in documents:

    mots = doc["text"].split()

    start = 0

    while start < len(mots):

        end = start + chunk_size

        chunk_text = " ".join(mots[start:end])

        chunks.append({
            "text": chunk_text,
            "source": doc["source"],
            "page": doc["page"]
        })

        start += chunk_size - chunk_overlap

print("Découpage terminé !")
print("Nombre total de chunks :", len(chunks))


Découpage terminé !
Nombre total de chunks : 14


4. Construire la recherche


Étape 4.1 - Création des embeddings

Dans cette étape, nous allons transformer les 64 chunks en vecteurs numériques à l'aide du modèle `all-MiniLM-L6-v2`.

Ces vecteurs permettent de représenter le sens des textes et de retrouver les passages les plus proches d'une question.

In [19]:
from sentence_transformers import SentenceTransformer

# Charger le modèle d'embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extraire le texte de chaque chunk
texts = [chunk["text"] for chunk in chunks]

# Transformer les chunks en vecteurs
embeddings = model.encode(texts)

print("Embeddings créés avec succès !")
print("Nombre de vecteurs :", len(embeddings))
print("Dimensions d'un vecteur :", embeddings.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings créés avec succès !
Nombre de vecteurs : 14
Dimensions d'un vecteur : 384


4.2 - Construction de la base vectorielle avec Chroma

Nous allons utiliser Chroma pour stocker les embeddings des chunks. Cette base vectorielle permettra de retrouver rapidement les passages les plus pertinents pour une question.

In [22]:

import chromadb

# Créer ou ouvrir la base vectorielle
client = chromadb.PersistentClient(path="../chroma_db")

# Ouvrir la collection
collection = client.get_or_create_collection(
    name="pfe_navigator"
)

# Supprimer les anciens chunks
anciens_ids = collection.get()["ids"]

if anciens_ids:
    collection.delete(ids=anciens_ids)

# Préparer les nouveaux identifiants
ids = [f"chunk_{i}" for i in range(len(chunks))]

metadatas = [
    {
        "source": chunk["source"],
        "page": chunk["page"]
    }
    for chunk in chunks
]

# Enregistrer les nouveaux chunks et leurs embeddings
collection.upsert(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Base vectorielle mise à jour avec succès !")
print("Nombre de chunks enregistrés :", collection.count())


Base vectorielle mise à jour avec succès !
Nombre de chunks enregistrés : 14


4.3 - Test de la recherche des passages pertinents

Dans cette étape, nous allons tester la recherche vectorielle. Le système transforme la question en embedding, puis recherche les 3 chunks les plus proches dans la base Chroma.


In [23]:
# Question de test
question = "Quelles sont les consignes à respecter pendant la soutenance ?"

# Transformer la question en embedding
question_embedding = model.encode([question])[0]

# Rechercher les 3 passages les plus proches
resultats = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

# Afficher les passages retrouvés
for i in range(len(resultats["documents"][0])):

    print("Passage", i + 1)
    print("Source :", resultats["metadatas"][0][i]["source"])
    print("Page :", resultats["metadatas"][0][i]["page"])
    print("Texte :", resultats["documents"][0][i])
    print("--------------------------------")

Passage 1
Source : 01_guide_pfe.pdf
Page : 1
Texte : d'accueil, ayant au moins 3 ans d'expérience. Il définit les tâches au quotidien et évalue l'étudiant.
--------------------------------
Passage 2
Source : 01_guide_pfe.pdf
Page : 2
Texte : Page 2 L'étudiant doit envoyer à son encadrant académique un point d'avancement écrit toutes les deux semaines. 7. Documents à fournir pendant le PFE • La convention de PFE, signée par les trois parties (étudiant, entreprise, ISIT) avant le démarrage. • Un rapport de mi-parcours de 5 à 10 pages, à rendre avant le 15 avril 2027. • Le pré-rapport complet, à rendre avant le 24 mai 2027 pour relecture par l'encadrant. • exemplaire relié remis au secrétariat du département. • L'attestation de fin de stage délivrée par l'entreprise. 8. Autorisation de soutenance L'étudiant est autorisé à soutenir si son encadrant académique donne un avis favorable écrit sur le rapport, si le taux de similarité du rapport ne dépasse pas 15 %, et si tous les documents sont

Observation :

La recherche vectorielle a retrouvé trois passages provenant de différents documents. Le premier passage contient des informations pertinentes sur les conditions de soutenance. Cependant, les deux autres passages ne répondent pas directement à la question.

## 5. Génération de la réponse avec Gemini

Dans cette étape, nous allons utiliser Gemini pour générer une réponse à partir des passages retrouvés dans notre base vectorielle Chroma. Le système doit répondre uniquement à partir des documents fournis.


### 5.1 Connexion à Gemini

Dans cette étape, nous établissons une connexion avec l'API Gemini afin d'utiliser le modèle pour générer des réponses à partir des passages retrouvés dans notre base documentaire.


In [12]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Trouver le dossier principal du projet
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

# Charger le fichier .env
env_path = project_root / "src" / ".env"
load_dotenv(dotenv_path=env_path, override=True)

# Récupérer la clé API
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Clé API Gemini introuvable !")

# Créer le client Gemini
client = genai.Client(api_key=api_key)

print("Connexion à Gemini réussie !")

Connexion à Gemini réussie !


### 5.2 Fonction de génération de la réponse

Dans cette étape, nous créons une fonction qui transmet à Gemini la question de l'utilisateur et les trois passages retrouvés par Chroma. Gemini utilise ces informations pour générer une réponse basée uniquement sur les documents fournis.

In [19]:

def generer_reponse(question, resultats):

    passages = resultats["documents"][0]
    metadonnees = resultats["metadatas"][0]

    contexte = ""

    for i in range(len(passages)):
        contexte += f"""
Passage {i + 1}
Source : {metadonnees[i]['source']}
Page : {metadonnees[i]['page']}
Contenu : {passages[i]}
"""

    prompt = f"""
Tu es un assistant spécialisé dans les documents relatifs au PFE et au stage.

Réponds uniquement à partir du contexte fourni.
N'utilise pas tes connaissances générales.
Si la réponse ne figure pas dans le contexte, dis :
"Je ne sais pas d'après les documents fournis."

Contexte :
{contexte}

Question :
{question}

Réponse :
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return response.text


### 5.3 Test du système RAG

Dans cette étape, nous posons une question sur les documents. Le système recherche les trois passages les plus pertinents dans Chroma, puis transmet ces passages à Gemini pour générer une réponse. Les sources utilisées sont également affichées.

In [7]:
import chromadb

# Se connecter à la base vectorielle Chroma
client_chroma = chromadb.PersistentClient(path="../chroma_db")

# Récupérer la collection existante
collection = client_chroma.get_collection(name="pfe_navigator")

print("Connexion à Chroma réussie !")
print("Nombre de chunks :", collection.count())

Connexion à Chroma réussie !
Nombre de chunks : 14


In [3]:
from sentence_transformers import SentenceTransformer

# Charger le modèle d'embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Modèle chargé avec succès !")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modèle chargé avec succès !


In [6]:
import chromadb

# Se connecter à la base Chroma
client_chroma = chromadb.PersistentClient(path="../chroma_db")

# Récupérer la collection existante
collection = client_chroma.get_collection(name="pfe_navigator")

print("Connexion à Chroma réussie !")
print("Nombre de chunks :", collection.count())

Connexion à Chroma réussie !
Nombre de chunks : 14


In [7]:
# Poser une question
question = "Quelles sont les consignes à respecter pendant la soutenance ?"

# Transformer la question en embedding
question_embedding = model.encode([question])[0]

# Rechercher les 3 passages les plus pertinents
resultats = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

# Afficher les passages retrouvés
print("Question :", question)
print("\nPassages retrouvés :")

for i in range(len(resultats["documents"][0])):
    print("\nPassage", i + 1)
    print("Source :", resultats["metadatas"][0][i]["source"])
    print("Page :", resultats["metadatas"][0][i]["page"])
    print("Texte :", resultats["documents"][0][i])
    print("-----------------------------")

Question : Quelles sont les consignes à respecter pendant la soutenance ?

Passages retrouvés :

Passage 1
Source : 01_guide_pfe.pdf
Page : 1
Texte : d'accueil, ayant au moins 3 ans d'expérience. Il définit les tâches au quotidien et évalue l'étudiant.
-----------------------------

Passage 2
Source : 01_guide_pfe.pdf
Page : 2
Texte : Page 2 L'étudiant doit envoyer à son encadrant académique un point d'avancement écrit toutes les deux semaines. 7. Documents à fournir pendant le PFE • La convention de PFE, signée par les trois parties (étudiant, entreprise, ISIT) avant le démarrage. • Un rapport de mi-parcours de 5 à 10 pages, à rendre avant le 15 avril 2027. • Le pré-rapport complet, à rendre avant le 24 mai 2027 pour relecture par l'encadrant. • exemplaire relié remis au secrétariat du département. • L'attestation de fin de stage délivrée par l'entreprise. 8. Autorisation de soutenance L'étudiant est autorisé à soutenir si son encadrant académique donne un avis favorable écrit sur le 

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Modèle chargé avec succès !")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modèle chargé avec succès !


In [11]:
import chromadb

# Connexion à la base vectorielle
client_chroma = chromadb.PersistentClient(path="../chroma_db")

# Récupérer la collection existante
collection = client_chroma.get_collection(name="pfe_navigator")

print("Connexion à Chroma réussie !")
print("Nombre de chunks :", collection.count())

Connexion à Chroma réussie !
Nombre de chunks : 14


In [22]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# Trouver le dossier du projet
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

# Charger la clé API depuis src/.env
env_path = project_root / "src" / ".env"
load_dotenv(dotenv_path=env_path, override=True)

# Récupérer la clé API
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Clé API Gemini introuvable !")

# Créer le client Gemini
client = genai.Client(api_key=api_key)

print("Client Gemini créé avec succès !")

Client Gemini créé avec succès !


In [23]:

question = "Quelles sont les consignes à respecter pendant la soutenance ?"

# Transformer la question en vecteur
question_embedding = model.encode([question])[0]

# Rechercher les 3 passages les plus pertinents dans Chroma
resultats = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

# Générer une réponse avec Gemini
reponse = generer_reponse(question, resultats)

# Afficher la réponse
print("Question :", question)
print("\nRéponse du système RAG :")
print(reponse)

# Afficher les sources utilisées
print("\nSources utilisées :")
for meta in resultats["metadatas"][0]:
    print("-", meta["source"], "page", meta["page"])


Question : Quelles sont les consignes à respecter pendant la soutenance ?

Réponse du système RAG :
D'après les documents fournis, les consignes et règles concernant le déroulement de la soutenance sont les suivantes :

- **Présence :** L'étudiant doit être présent dans la salle 15 minutes avant son heure de passage.
- **Durée totale :** La soutenance dure 45 minutes au total, découpée comme suit :
  1. **Présentation de l'étudiant :** 20 minutes, sans interruption.
  2. **Questions du jury et discussion :** 20 minutes.
  3. **Délibération à huis clos :** environ 5 minutes, pendant lesquelles l'étudiant doit quitter la salle.

Sources utilisées :
- 01_guide_pfe.pdf page 1
- 01_guide_pfe.pdf page 2
- 04_guide_soutenance.pdf page 1


## 6. Test d'une question sans réponse dans le corpus


Dans cette étape, nous allons poser une question dont la réponse ne figure pas dans les documents. Le système doit reconnaître qu'il ne possède pas l'information et répondre qu'il ne sait pas.

In [25]:
question = "Quelle est la couleur préférée du directeur de l'établissement ?"

question_embedding = model.encode([question])[0]

resultats = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

reponse = generer_reponse(question, resultats)

print("Question :", question)
print("\nRéponse du système RAG :")
print(reponse)

Question : Quelle est la couleur préférée du directeur de l'établissement ?

Réponse du système RAG :
Je ne sais pas d'après les documents fournis.


## 7. Comparaison entre Gemini seul et le système RAG

Dans cette étape, nous allons comparer les réponses de Gemini seul avec celles du système RAG pour les cinq questions préparées. L'objectif est d'évaluer l'apport des documents dans la qualité et la précision des réponses.


### 7.1 Tableau de comparaison

Le tableau suivant permet de comparer les réponses de Gemini seul avec celles du système RAG. Nous évaluons leur précision et leur capacité à répondre à partir des documents fournis.



| Question | Gemini seul | Système RAG | Meilleur résultat |
|---|---|---|---|
| Question 1:Quelles sont les conditions et la durée d'inscription au PFE ? | | |  |
| Question 2:Quelles sont les parties obligatoires du rapport et dans quel ordre ?|  |  |  |
| Question 3:Comment se déroule la soutenance et quelle est sa durée ? |  |  |  |
| Question 4:Quelles sont les obligations de l'étudiant pendant le stage ?|  |  |  |
| Question 5:Quel est le montant mensuel de la bourse d'excellence ? (sans réponse) |  |  |  |
